# 05 · Interpretable-by-design fix (Track D)

Rules: `Docs/PREREGISTRATION.md` section 10 (frozen 2026-11-15) and `Docs/PREREG_ADDENDUM_TRACK_D.md`, which must be committed before this notebook trains anything. The candidates are **not chosen here**: Cell 2 reads the saved verdicts of notebooks 03 and 04 and applies the decision table of section 10.

Two stages, chosen by `STAGE` in Cell 2. Each run goes top to bottom.

| `STAGE` | Data | What it does | Then |
|---|---|---|---|
| `"VAL"` | FIT, VAL | Trains the candidates that need training (5 seeds each, plus the shuffled-label control), then the four-item scorecard on VAL. Writes `reports/05_val.json` | Send me the output of Cells 4–9 |
| `"CONF_B"` | CONF-B | Items 1–3 recomputed **once** on CONF-B with everything fixed. Writes `reports/05_summary.json` and `reports/05_confB_done.json` | Notebook 04 with `STAGE = "EXT"` |

Scorecard (section 10). Item 1 is mandatory; the winner has the most of items 2–4.

1. **Performance**: gate D1 passed, and AUROC not more than 0.01 below the softmax model of the same seed.
2. **Sanity**: the candidate's primary explanation passes the screen of section 5 against its own untrained and shuffled-label controls, in 5 of 5 seeds.
3. **Faithfulness**: deleting the top 20 % of hours by the explanation lowers the t+6 logit more than deleting random hours, and more than raw attention does in the standard softmax model.
4. **Stability**: the explanation ranks hours similarly across the 5 seeds (mean Spearman ≥ 0.5, and at least that of raw attention in the standard model).

In smoke mode every candidate is switched on (so every code path runs), CONF-B is never loaded (the dry run uses a second draw of VAL stays), and no completion marker is written. Aggregates only: nothing here prints row-level patient data.

In [ ]:
# ============================ CELL 1 · BOOTSTRAP: clone the repo, import sepsis_core from it ============================
REPO_URL = "https://github.com/Myakey/sepsis-transformer-analysis.git"
REPO_REF = "main"                                  # branch, tag or commit hash. PIN A COMMIT HASH for any run you will report.
REPO_DIR = "/content/sepsis-transformer-analysis"

import os, sys, subprocess
sys.dont_write_bytecode = True
if "google.colab" in sys.modules and not os.path.exists("/content/drive/MyDrive"):      # Drive holds the data and the checkpoints
    from google.colab import drive
    drive.mount("/content/drive")

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(f"git {' '.join(args[:2])} failed: {r.stderr.strip()[-300:]}")
    return r.stdout.strip()

def _auth_url(url):
    tok = None                                     # private repo: Colab secret GITHUB_TOKEN (Notebook access on)
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
    except Exception:
        tok = os.environ.get("GITHUB_TOKEN")
    return url.replace("https://", f"https://{tok}@") if tok else url

if not os.path.isdir(os.path.join(REPO_DIR, ".git")): _git("clone", "--quiet", _auth_url(REPO_URL), REPO_DIR)
else: _git("fetch", "--quiet", "--all", cwd=REPO_DIR)
_git("checkout", "--quiet", REPO_REF, cwd=REPO_DIR)
if REPO_REF in _git("branch", "--format=%(refname:short)", cwd=REPO_DIR).split(): _git("pull", "--quiet", "--ff-only", cwd=REPO_DIR)
_git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)        # never leave a token in the clone's config
CORE_COMMIT = _git("rev-parse", "HEAD", cwd=REPO_DIR)
os.environ["SEPSIS_PROJECT_ROOT"] = REPO_DIR
os.environ["SEPSIS_CORE_COMMIT"] = CORE_COMMIT
for m in [k for k in sys.modules if k == "sepsis_core" or k.startswith("sepsis_core.")]: del sys.modules[m]
if os.path.join(REPO_DIR, "Notebooks", "v4") not in sys.path: sys.path.insert(0, os.path.join(REPO_DIR, "Notebooks", "v4"))

import importlib.util
_need = [p for p in ("mlflow", "lightgbm") if importlib.util.find_spec(p) is None]
if _need: subprocess.call([sys.executable, "-m", "pip", "install", "-q", *_need])

import json, time, gc, hashlib, pickle
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import sepsis_core
from sepsis_core import config as C, data, dataset, models, engine, metrics, calibration, controls, runs, plots, tracking, attention, physionet, correlational, causal, explain, gbdt, fix
print("sepsis_core", sepsis_core.__version__, "@ commit", CORE_COMMIT[:12], f"({REPO_REF})", "| torch", torch.__version__)

In [ ]:
# ============================ CELL 2 · SWITCHES, AND THE DECISION TABLE READ FROM THE RESULTS OF 03 AND 04 ============================
SMOKE_TEST = False
STAGE = "VAL"            # "VAL" -> "CONF_B". One stage per run, top to bottom.
OPT = dict(train=True, kernel_second=True, n_points=1000, methods_table=True, variable_table=True)
FORCE = False            # True = recompute cached screens and explanations of the current stage (never retrains a finished model)

runs.setup(smoke=SMOKE_TEST); fix.install()
if STAGE not in ("VAL", "CONF_B"): raise ValueError("STAGE must be 'VAL' or 'CONF_B'")
if not data.FoldGuard("fix").prereg_frozen(): raise PermissionError("Docs/PREREGISTRATION.md and its .frozen stamp are not both in this commit.")
ADDENDUM = os.path.join(C.PATHS.DOCS_DIR, "PREREG_ADDENDUM_TRACK_D.md")
if not os.path.exists(ADDENDUM): raise PermissionError("Docs/PREREG_ADDENDUM_TRACK_D.md is not in this commit. Section 10: the candidates, their primary explanations and their "
                                                       "hyperparameters are committed before any 05 training. Commit the file, push, re-run Cell 1.")
ADD_SHA = hashlib.sha256(open(ADDENDUM, "rb").read().replace(b"\r\n", b"\n")).hexdigest()
SEEDS = list(C.CFG.SEEDS)[: (2 if SMOKE_TEST else 5)]; PRIMARY = SEEDS[0]; HP = dict(C.V3_BEST_HP); REP = runs.report_path; K6 = 2      # K6: index of the t+6 logit
OUT = os.path.join(runs.run_root(), "reports", "05"); TD = os.path.join(runs.run_root(), "trackD"); os.makedirs(OUT, exist_ok=True)
N_POINTS = 60 if SMOKE_TEST else int(OPT["n_points"])

for f_, nb_ in (("03_summary.json", "03"), ("04_summary.json", "04 (stage CONF_A)")):
    if not os.path.exists(REP(f_)): raise FileNotFoundError(f"{REP(f_)} is missing. Run notebook {nb_} first" + (" with SMOKE_TEST = True" if SMOKE_TEST else "") + ": the decision table is read from it.")
S03 = json.load(open(REP("03_summary.json"))); S04 = json.load(open(REP("04_summary.json")))
if not SMOKE_TEST and (S04.get("smoke") or S03.get("smoke")): raise RuntimeError("reports/03_summary.json or 04_summary.json in this run folder comes from a smoke run.")
TABLE = fix.decision_table(S03, S04, PRIMARY); CAND = fix.CANDIDATES
CANDS = fix.candidates(TABLE, kernel_second=OPT["kernel_second"], force_all=SMOKE_TEST)
ev = TABLE["evidence"]
print(f"stage: {STAGE} | seeds: {SEEDS} | addendum sha256 {ADD_SHA[:12]}")
print("DECISION TABLE (section 10), read from the saved results:")
print(f"   row 1  softmax rung-0 verdict (03) = {ev['softmax_rung0_verdict']}  -> {'FIRES: variable-level tokens; kernel attention second' if TABLE['rows'][1] else 'does not fire'}")
print(f"   row 2  four-outcome cell (04): softmax {ev['four_outcome']['softmax']} / sigmoid {ev['four_outcome']['sigmoid']}, cell assigned: {ev['four_outcome']['cell'] is not None} | "
      f"softmax heads with > 5% of weights below 1e-6, primary seed (03): {ev['softmax_primary_seed_heads_with_floor_fraction_above_5pct']}  -> {'FIRES: sigmoid as the explanation layer' if TABLE['rows'][2] else 'does not fire'}")
print(f"   row 3  softmax rungs (04, floor in force: {ev['iia_floor_in_force']}): {ev['softmax_rungs']}  -> {'FIRES: no new model; row-centred logit and norm-weighted attention' if TABLE['rows'][3] else 'does not fire'}")
print(f"   row 4  S5 trigger (03) = {ev['S5_trigger']}  -> {'FIRES: every explanation is reported with seed and bootstrap bands' if TABLE['rows'][4] else 'does not fire (bands are reported anyway)'}")
if SMOKE_TEST: print("SMOKE: every candidate is switched on regardless of the table.")
print("candidates:", {c: CAND[c]["label"] for c in CANDS})
if not CANDS: raise RuntimeError("No row of the decision table fired, so section 10 names no candidate. Send me this output.")

def model_name(cand, seed): return f"{CAND[cand]['prefix']}_s{seed}"
def shuf_name(cand): return f"{CAND[cand]['prefix']}_shuf_v3"
def have(name, which="best.pt"): return os.path.exists(os.path.join(runs.model_dir(name), which))
def load(name, which="best.pt"): return runs.load_trained(name, B["store"], which)
def free():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
for seed in SEEDS:
    if not have(f"softmax_s{seed}"): raise FileNotFoundError(f"softmax_s{seed} is not in {runs.run_root()}/models. Run notebook 02 first.")

In [ ]:
# ============================ CELL 3 · DATA: FIT/VAL, the LightGBM reference of the gate, severity (analysis only), the analysis stays of 03 ============================
B = runs.prepare_mimic(stage="fix"); st = B["store"]
G = gbdt.fit_or_load(runs.model_dir("gbdt_iculos"), st, B["pre"], B["fit_idx"], B["val_idx"], with_iculos=True)
G_ROWS, G_LOGIT = G["rows"], gbdt.to_logit(G["scores"])
SEV_ALL, SUBJ = correlational.load_severity(st, B["val_idx"], [C.CFG.CORR_SEVERITY_COL]); SEV = SEV_ALL[list(SEV_ALL)[0]]

def make_set(tag, store, sev, subj_map, pool, splits, pts_seed):
    """One evaluation set: the screen of section 5 on its DISCOVERY / SCREEN stays, and the readouts (stay, hour) every explanation is scored on."""
    pool = np.asarray(pool, np.int64); subj_arr = np.full(len(store.starts), -1, np.int64)
    for i in pool: subj_arr[int(i)] = int(subj_map.get(int(store.ids[store.starts[int(i)]]), -1))
    scr = correlational.Screen(store, sev, subj_map, splits, B["pre"].feat)
    pts = explain.sample_points(store, pool, n=N_POINTS, per_stay=3, horizon_index=K6, subj_of_stay=subj_arr, seed=pts_seed)
    os.makedirs(os.path.join(TD, tag), exist_ok=True)
    S = dict(tag=tag, store=store, screen=scr, pts=pts, pool=pool, pts_id=hashlib.sha1(pts["a"].tobytes() + pts["t"].tobytes()).hexdigest()[:12])
    print(f"[{tag}] stays {len(pool):,} | screen: DISCOVERY {len(splits['DISCOVERY'])}, SCREEN {len(splits['SCREEN'])} | readouts {len(pts['a']):,} from {len(np.unique(pts['stay'])):,} stays "
          f"({len(np.unique(pts['subj'])):,} subjects), id {S['pts_id']}")
    return S

VAL = None
if STAGE == "VAL":
    if SMOKE_TEST: SPLITS, SPLIT_INFO = correlational.draw_splits(st, SEV, B["val_idx"], SUBJ)
    else:          SPLITS, SPLIT_INFO = correlational.v3_splits(st)          # the same stays as notebook 03
    VAL = make_set("VAL", st, SEV, SUBJ, B["val_idx"], SPLITS, C.CFG.SEED + 500)

## Training (stage VAL only): the candidates that need a new model

Same data, loss, schedule, early stopping and checkpoint soup as the primary arms of notebook 02; hyperparameters are the matched softmax model's, with no tuning. Finished models are skipped, so this cell can be re-run after a disconnect.

In [ ]:
# ============================ CELL 4 · TRAIN ============================
REC = {}
if STAGE == "VAL" and OPT["train"]:
    for c in CANDS:
        if not CAND[c]["trains"]: continue
        for seed in SEEDS:
            nm = model_name(c, seed); REC[nm] = runs.train_model(B, nm, fix.hp_for(c, HP), seed=seed, role=f"trackD_{c}")
    todo = [c for c in CANDS if CAND[c]["trains"] and not have(shuf_name(c))]
    if todo:
        ST_SHUF, _rng = controls.shuffled_store(st, B["fit_idx"]); SFIT = controls.v3_fit_subset(_rng, B["fit_idx"], 0.3)       # v3 protocol, same random stream as 02
        for c in todo:
            runs.train_model(B, shuf_name(c), fix.hp_for(c, HP), seed=C.CFG.SEED + 5, role="control_shuffled_v3protocol", store=ST_SHUF, fit_idx=SFIT,
                             max_epochs=(1 if SMOKE_TEST else 4), patience=5, soup_k=0)
        del ST_SHUF; gc.collect()
for c in CANDS:
    for nm, wh in [(model_name(c, s), w) for s in SEEDS for w in ("best.pt", "init.pt")] + [(shuf_name(c), "best.pt")]:
        if not have(nm, wh): raise FileNotFoundError(f"model '{nm}' ({wh}) is not in {runs.run_root()}/models. Run this notebook with STAGE = 'VAL' and OPT['train'] = True first.")
if "vartok" in CANDS:
    a_ = torch.load(os.path.join(runs.model_dir(f"vartok_s{PRIMARY}"), "init.pt"), map_location="cpu", weights_only=False)["model"]
    b_ = torch.load(os.path.join(runs.model_dir(f"softmax_s{PRIMARY}"), "init.pt"), map_location="cpu", weights_only=False)["model"]
    shared = [k for k in b_ if "base." + k in a_]
    print(f"vartok_s{PRIMARY}: trunk starts from the initial weights of softmax_s{PRIMARY}: {all(torch.equal(a_['base.' + k], b_[k]) for k in shared)} ({len(shared)} shared tensors)")
    del a_, b_
print("models present:", {c: [model_name(c, s) for s in SEEDS] for c in CANDS})

In [ ]:
# ============================ CELL 5 · HELPERS: the four scorecard items, written once and used on VAL and on CONF-B ============================
PLAN_T = {"DISCOVERY": ["real", "circ_shift", "donor", "rand_proj"], "SCREEN": ["real"]}; PLAN_C = {"DISCOVERY": ["real"], "SCREEN": ["real"]}
SHUF_RUNS = {}

def scores_on(S, name):
    """Logits of a model on the stays of an evaluation set (VAL: the scores saved at training time)."""
    if S["tag"] == "VAL": return runs.val_scores(name)
    p = os.path.join(TD, S["tag"], f"scores__{name}.npz")
    if os.path.exists(p) and not FORCE:
        z = np.load(p); return z["rows"], z["logits"]
    m = load(name); rows, lg = runs.predict_store(m, S["store"], S["pool"]); del m; free()
    np.savez_compressed(p, rows=rows, logits=lg.astype(np.float32)); return rows, lg

def gbdt_on(S):
    if S["tag"] == "VAL": return G_ROWS, G_LOGIT
    rows, p = gbdt.predict(G, S["store"], B["pre"], S["pool"]); o = np.argsort(rows, kind="stable"); return rows[o], gbdt.to_logit(p[o])

def performance(S, cand):
    """Item 1: gate D1 at the primary seed, and AUROC(candidate) - AUROC(softmax, same seed) per seed (stay-level paired bootstrap, mean of horizons)."""
    out = dict(by_seed=[])
    for seed in SEEDS:
        rows, lg = scores_on(S, model_name(cand, seed)); rows_s, lg_s = scores_on(S, f"softmax_s{seed}")
        if not np.array_equal(rows, rows_s): raise RuntimeError(f"[{S['tag']}] {model_name(cand, seed)} and softmax_s{seed} were scored on different rows")
        ni = explain.noninferiority(S["store"], rows, lg, lg_s, n_boot=runs.STATE["n_boot"]); ni["seed"] = int(seed); out["by_seed"].append(ni)
        if seed == PRIMARY:
            gr, gl = gbdt_on(S)
            if not np.array_equal(rows, gr): raise RuntimeError(f"[{S['tag']}] the LightGBM reference was scored on different rows")
            out["gate"] = metrics.gate(S["store"], rows, lg, gl, n_boot=runs.STATE["n_boot"]); out["auroc_primary"] = float(metrics.compute_metrics(S["store"], rows, lg)["auroc_mean"])
    return out

def screen_run(S, name, which, arm, plan, expose):
    m = load(name, which); fix.set_expose(m, expose, K6); t0 = time.time(); r = S["screen"].run(m, arm, plan); del m; free()
    print(f"    [{S['tag']}] captured {name} ({which}) in {time.time() - t0:.0f}s"); return r

def screen_record(S, cand, seed):
    """Item 2: the screen of section 5 for one trained model against its own untrained network and the v3-protocol shuffled-label model.
    On VAL, models that notebook 03 already screened are read from its saved records (same stays, same code)."""
    c = CAND[cand]; name = model_name(cand, seed); path = os.path.join(OUT, f"screen__{S['tag']}__{c['prefix']}{'_' + c['expose'] if c['expose'] else ''}_s{seed}.json")
    if os.path.exists(path) and not FORCE: return json.load(open(path))
    p03 = os.path.join(runs.run_root(), "reports", "03", f"{name}__best.json")
    if S["tag"] == "VAL" and os.path.exists(p03): full = json.load(open(p03)); src = "notebook 03"
    else:
        sk = (S["tag"], shuf_name(cand), c["expose"])
        if sk not in SHUF_RUNS: SHUF_RUNS[sk] = screen_run(S, shuf_name(cand), "best.pt", c["arm"], PLAN_C, c["expose"])
        roles = dict(trained=screen_run(S, name, "best.pt", c["arm"], PLAN_T, c["expose"]), init=screen_run(S, name, "init.pt", c["arm"], PLAN_C, c["expose"]), shuf_v3=SHUF_RUNS[sk])
        full = correlational.analyse(roles, c["arm"]); src = "computed here"; del roles
    rec = dict(model=name, source=src, L=full["L"], H=full["H"], by_dv={dv: {k_: d[k_] for k_ in ("band", "threshold", "counts", "E", "max_abs_trained")} for dv, d in full["by_dv"].items()})
    json.dump(rec, open(path, "w"), default=runs._js); return rec

def explained(S, name, method):
    """Items 3 and 4 for one model: its explanation on every readout, deletion / insertion along it, and along a random order (same model, same readouts)."""
    p = os.path.join(TD, S["tag"], f"explain__{name}__{method}.pkl")
    if os.path.exists(p) and not FORCE:
        d = pickle.load(open(p, "rb"))
        if d.get("pts_id") == S["pts_id"]: return d
    m = load(name); t0 = time.time(); sc = fix.scores(m, S["store"], S["pts"], method, horizon_index=K6)
    F = explain.faithfulness(m, S["store"], S["pts"], sc, horizon_index=K6)
    Fr = explain.faithfulness(m, S["store"], S["pts"], explain.random_scores(S["pts"], seed=C.CFG.SEED + 77), horizon_index=K6)
    del m; free(); d = dict(scores=sc, F=F, F_rand=Fr, pts_id=S["pts_id"]); pickle.dump(d, open(p, "wb"))
    print(f"    [{S['tag']}] explained {name} with {method} in {time.time() - t0:.0f}s"); return d

def pooled(ds, which, key="comprehensiveness"): return np.mean([d[which][key] for d in ds], 0)      # per readout, mean over the seeds

def faith_stab(S, cand):
    c = CAND[cand]; g = S["pts"]["subj"]
    ds = [explained(S, model_name(cand, s), c["method"]) for s in SEEDS]; ref = [explained(S, f"softmax_s{s}", "raw_attention") for s in SEEDS]
    comp = pooled(ds, "F"); m_, lo_, hi_ = causal.boot_mean(comp, g)
    per_seed = [float(np.mean(d["F"]["comprehensiveness"])) for d in ds]
    return dict(comprehensiveness=dict(mean=float(m_), lo=float(lo_), hi=float(hi_), per_seed=per_seed),
                random_order=float(np.mean(pooled(ds, "F_rand"))), raw_attention_standard_model=float(np.mean(pooled(ref, "F"))),
                vs_random=explain.margin(dict(comprehensiveness=comp), dict(comprehensiveness=pooled(ds, "F_rand")), g),
                vs_raw_attention=explain.margin(dict(comprehensiveness=comp), dict(comprehensiveness=pooled(ref, "F")), g),
                primary_seed=explain.summary(ds[0]["F"], g),
                stability=float(explain.seed_stability({s: d["scores"] for s, d in zip(SEEDS, ds)})["mean"]),
                stability_raw_attention=float(explain.seed_stability({s: d["scores"] for s, d in zip(SEEDS, ref)})["mean"]))

def card_for(S, cand, stability=None):
    """The scorecard of one candidate on one evaluation set. `stability` = (candidate, raw attention) taken from VAL when the set is CONF-B (item 4 is not recomputed there)."""
    c = CAND[cand]; P = performance(S, cand); prim = next(x for x in P["by_seed"] if x["seed"] == PRIMARY)
    E = [screen_record(S, cand, s)["by_dv"][c["dv"]]["E"] for s in SEEDS]; verdict = correlational.seed_verdict(E)
    FS = faith_stab(S, cand); sc_, sr_ = (FS["stability"], FS["stability_raw_attention"]) if stability is None else stability
    card = explain.scorecard(cand, P["gate"]["passed"], prim, P["by_seed"], verdict, FS["vs_random"], FS["vs_raw_attention"], sc_, sr_)
    card["detail"].update(set=S["tag"], explanation=c["method"], screen_dv=c["dv"], E_per_seed=E, gate=P["gate"], auroc_primary=P["auroc_primary"],
                          difference_per_seed=[x["d"] for x in P["by_seed"]], faithfulness=FS, stability_measured_on_this_set=FS["stability"])
    return card

def show(card):
    d = card["detail"]; f = d["faithfulness"]; ni = d["noninferiority_primary"]; yn = lambda b: "PASS" if b else "fail"
    print(f"\n{card['candidate']}: {CAND[card['candidate']]['label']}   [{d['set']}]")
    print(f"   1 performance   {yn(card['performance'])} | gate D1 {'passed' if d['gate']['passed'] else 'FAILED'} | AUROC {d['auroc_primary']:.4f} | minus softmax, primary seed {ni['d']:+.4f} [{ni['lo']:+.4f}, {ni['hi']:+.4f}] | "
          f"mean over seeds {d['mean_difference_over_seeds']:+.4f} (per seed {[round(x, 4) for x in d['difference_per_seed']]})")
    print(f"   2 sanity        {yn(card['sanity'])} | explanation '{d['explanation']}', screen DV '{d['screen_dv']}': excess E per seed {d['E_per_seed']} -> {d['sanity_verdict']}")
    print(f"   3 faithfulness  {yn(card['faithfulness'])} | comprehensiveness {f['comprehensiveness']['mean']:+.4f} [{f['comprehensiveness']['lo']:+.4f}, {f['comprehensiveness']['hi']:+.4f}] "
          f"(seed range {min(f['comprehensiveness']['per_seed']):+.4f} to {max(f['comprehensiveness']['per_seed']):+.4f}) | random order {f['random_order']:+.4f} | raw attention, standard model {f['raw_attention_standard_model']:+.4f}")
    print(f"                   minus random {f['vs_random']['mean']:+.4f} [{f['vs_random']['lo']:+.4f}, {f['vs_random']['hi']:+.4f}] | minus raw attention {f['vs_raw_attention']['mean']:+.4f} "
          f"[{f['vs_raw_attention']['lo']:+.4f}, {f['vs_raw_attention']['hi']:+.4f}] | sufficiency, primary seed {f['primary_seed']['sufficiency']['mean']:+.4f}")
    print(f"   4 stability     {yn(card['stability'])} | mean pairwise Spearman across seeds {d['stability']:.3f} | raw attention, standard model {d['stability_raw_attention']:.3f} | needs >= {C.CFG.STABILITY_MIN} and >= raw attention"
          + ("" if d["set"] == "VAL" else f"   (taken from VAL; measured on this set: {d['stability_measured_on_this_set']:.3f})"))
    print(f"   eligible: {card['eligible']} | items 2-4 passed: {card['score']}")

## Stage VAL: the scorecard

Comprehensiveness is the drop of the t+6 logit when the top 20 % of hours by the explanation are set to "not measured"; it is averaged over the five seeds per readout, then bootstrapped over subjects. Stability is the mean pairwise Spearman correlation of the hour ranking across seeds on the same readouts.

In [ ]:
# ============================ CELL 6 · VAL: the scorecard of every candidate ============================
CARDS = {}
if STAGE == "VAL":
    if "vartok" in CANDS:
        m_ = load(f"vartok_s{PRIMARY}"); err = fix.additivity_error(m_, st, VAL["pts"], horizon_index=K6); del m_; free()
        print(f"vartok_s{PRIMARY}: | bias + sum of hour contributions - logit | <= {err:.2e} on 16 readouts (the model is additive by construction; this must be ~1e-5)")
        if not err < 1e-3: raise RuntimeError("the additive decomposition of the VarTok model does not reproduce its logit: this is a bug, not a result. Send me this line.")
    for c in CANDS:
        CARDS[c] = card_for(VAL, c); show(CARDS[c])

In [ ]:
# ============================ CELL 7 · VAL: the winner, and what is fixed before CONF-B ============================
if STAGE == "VAL":
    WINNER = explain.pick_winner(list(CARDS.values()))
    print("WINNER on VAL (eligible candidate with the most of items 2-4; ties by the larger comprehensiveness margin over raw attention):", WINNER)
    for c in CANDS: print(f"   {c:<14s} eligible {CARDS[c]['eligible']} | items 2-4 {CARDS[c]['score']} | sanity {CARDS[c]['sanity']} faithfulness {CARDS[c]['faithfulness']} stability {CARDS[c]['stability']}")
    if WINNER is None: print("No candidate is eligible (item 1 failed for all). S6 cannot fire; the CONF-B stage is still run so that 05 is complete, and the result is reported as it is.")
    VALOUT = dict(core_commit=CORE_COMMIT, run=C.CFG.RUN_NAME, smoke=SMOKE_TEST, seeds=SEEDS, addendum_sha256=ADD_SHA, decision_table=TABLE, candidates=CANDS, cards=CARDS, winner=WINNER,
                  readouts=dict(n=int(len(VAL["pts"]["a"])), id=VAL["pts_id"]), time=time.strftime("%Y-%m-%dT%H:%M:%S"))
    runs.save_json(VALOUT, "05_val.json")

In [ ]:
# ============================ CELL 8 · VAL, REPORTED BESIDE THE SCORECARD: the six explanation methods on every candidate (primary seed, 300 readouts) ============================
METHODS = {}
if STAGE == "VAL" and OPT["methods_table"]:
    sub = {k_: v[: min(300, len(v))] for k_, v in VAL["pts"].items()}
    for name in dict.fromkeys([f"softmax_s{PRIMARY}"] + [model_name(c, PRIMARY) for c in CANDS]):
        p = os.path.join(OUT, f"methods__{name}.json")
        if os.path.exists(p) and not FORCE: METHODS[name] = json.load(open(p))
        else:
            m_ = load(name); rows = {}
            for meth in fix.methods_for(m_) + ["random order"]:
                sc = explain.random_scores(sub, seed=C.CFG.SEED + 77) if meth == "random order" else fix.scores(m_, st, sub, meth, horizon_index=K6)
                s_ = explain.summary(explain.faithfulness(m_, st, sub, sc, horizon_index=K6), sub["subj"])
                rows[meth] = {k_: s_[k_] for k_ in ("comprehensiveness", "sufficiency", "deletion_auc", "insertion_auc")}
            del m_; free(); METHODS[name] = rows; json.dump(rows, open(p, "w"), default=runs._js)
        print(f"\n{name}: t+6 logit, {len(sub['a'])} readouts [mean, 95% CI over subjects]. Larger comprehensiveness and smaller sufficiency = more faithful.")
        for meth, r in METHODS[name].items():
            print(f"   {meth:<22s} comprehensiveness {r['comprehensiveness']['mean']:+.4f} [{r['comprehensiveness']['lo']:+.4f}, {r['comprehensiveness']['hi']:+.4f}] | sufficiency {r['sufficiency']['mean']:+.4f} | "
                  f"deletion AUC {r['deletion_auc']['mean']:+.4f} | insertion AUC {r['insertion_auc']['mean']:+.4f}")
    runs.save_json(METHODS, "05_methods.json")

In [ ]:
# ============================ CELL 9 · VAL, DESCRIPTIVE: which named variables the VarTok readout uses (aggregate over readouts and seeds) ============================
VARS = {}
if STAGE == "VAL" and OPT["variable_table"] and "vartok" in CANDS:
    tabs = []
    for seed in SEEDS:
        m_ = load(f"vartok_s{seed}"); tabs.append({r["variable"]: r for r in fix.variable_table(m_, st, VAL["pts"], B["pre"].feat, horizon_index=K6)}); del m_; free()
    for v in tabs[0]:
        VARS[v] = {k_: float(np.mean([t[v][k_] for t in tabs])) for k_ in ("attention_share", "mean_contribution", "mean_abs_contribution")}
        VARS[v]["rank_per_seed"] = [1 + sorted(t, key=lambda x: -t[x]["mean_abs_contribution"]).index(v) for t in tabs]
    print(f"VarTok, t+6 logit, {len(VAL['pts']['a'])} readouts, mean over {len(SEEDS)} seeds. Top 12 variables by mean |contribution| (summed over the hours visible at the readout):")
    for v in sorted(VARS, key=lambda x: -VARS[x]["mean_abs_contribution"])[:12]:
        print(f"   {v:<36s} share of attention {VARS[v]['attention_share']:.3f} | mean contribution {VARS[v]['mean_contribution']:+.4f} | mean |contribution| {VARS[v]['mean_abs_contribution']:.4f} | rank per seed {VARS[v]['rank_per_seed']}")
    runs.save_json(VARS, "05_vartok_variables.json")
if STAGE == "VAL":
    print("\nVAL stage complete. Send me the output of Cells 2, 4, 6, 7, 8 and 9 (all aggregate). CONF-B has not been loaded. Next: STAGE = 'CONF_B'.")

## Stage CONF-B: items 1–3 once, with everything fixed

Fixed before this stage: the candidates (decision table), their primary explanations, the models and their controls, the screen procedure and the readout sampling rule. On CONF-B the screen draws its DISCOVERY / SCREEN stays by v3's procedure and builds its own control band from the same controls. Item 4 (stability) is not recomputed (section 10); the VAL value is carried.

In [ ]:
# ============================ CELL 10 · CONF-B: open the fold (once) ============================
CONFB, V, SUMMARY = None, None, None
if STAGE == "CONF_B":
    if not os.path.exists(REP("05_val.json")): raise FileNotFoundError(f"{REP('05_val.json')} is missing: run STAGE = 'VAL' first.")
    V = json.load(open(REP("05_val.json")))
    if V["candidates"] != CANDS or V["addendum_sha256"] != ADD_SHA or bool(V["smoke"]) != bool(SMOKE_TEST):
        raise RuntimeError("the candidate list or the addendum changed after the VAL stage. CONF-B is opened only with what the VAL stage fixed; re-run STAGE = 'VAL' on this commit first.")
    if os.path.exists(REP("05_confB_done.json")) and not SMOKE_TEST:
        SUMMARY = json.load(open(REP("05_summary.json")))
        print("The CONF-B pass is already complete and is not recomputed. Its saved result:")
        print(json.dumps({k_: SUMMARY[k_] for k_ in ("winner_on_VAL", "agreement", "passes_scorecard_strict", "passes_sanity_at_noninferior_auroc", "S6_trigger")}, indent=1))
    elif SMOKE_TEST:
        sp2, _ = correlational.draw_splits(st, SEV, B["val_idx"], SUBJ, seed=C.CFG.SEED + 613)
        CONFB = make_set("VAL2", st, SEV, SUBJ, B["val_idx"], sp2, C.CFG.SEED + 614)
        print("SMOKE: CONF-B is not loaded. The dry run uses a second draw of VAL stays and readouts.")
    else:
        FB = causal.prepare_fold("CONF_B", B["pre"], guard=data.FoldGuard("fix"), stage="fix"); sb = FB["store"]
        subj_b = {int(s): int(u) for s, u in zip(sb.ids[sb.starts].tolist(), np.asarray(FB["subj"]).tolist())}; pool_b = np.arange(len(sb.starts))
        sp_b, info_b = correlational.draw_splits(sb, FB["sev"], pool_b, subj_b, seed=C.CFG.SEED + 613)
        print(f"CONF-B: subjects {FB['n_subjects']:,} | stays {FB['n_stays']:,} | subjects per fold in the split table: {FB['counts']} | stays eligible for the screen {info_b['n_eligible']:,}")
        CONFB = make_set("CONF_B", sb, FB["sev"], subj_b, pool_b, sp_b, C.CFG.SEED + 614)

In [ ]:
# ============================ CELL 11 · CONF-B: items 1-3, agreement with VAL, S6, SAVE (this is what to send back) ============================
if STAGE == "CONF_B" and CONFB is not None:
    CARDS_B = {}
    for c in CANDS:
        dv = V["cards"][c]["detail"]; CARDS_B[c] = card_for(CONFB, c, stability=(dv["stability"], dv["stability_raw_attention"])); show(CARDS_B[c])
    AGREE = {c: bool(explain.agrees_on_conf_b(V["cards"][c], CARDS_B[c])) for c in CANDS}
    STRICT = {c: bool(all(V["cards"][c][k_] for k_ in ("performance", "sanity", "faithfulness", "stability")) and all(CARDS_B[c][k_] for k_ in ("performance", "sanity", "faithfulness"))) for c in CANDS}
    SANITY = {c: bool(V["cards"][c]["performance"] and V["cards"][c]["sanity"] and CARDS_B[c]["performance"] and CARDS_B[c]["sanity"]) for c in CANDS}
    print("\nVAL vs CONF-B, items 1-3 (a candidate passes only if CONF-B agrees with VAL):")
    for c in CANDS:
        print(f"   {c:<14s} VAL {[bool(V['cards'][c][k_]) for k_ in ('performance', 'sanity', 'faithfulness')]} | CONF-B {[bool(CARDS_B[c][k_]) for k_ in ('performance', 'sanity', 'faithfulness')]} | agrees and item 1 holds: {AGREE[c]} | "
              f"passes all items (1-4 on VAL, 1-3 on CONF-B): {STRICT[c]} | passes the sanity check at non-inferior AUROC on both: {SANITY[c]}")
    S6 = bool(any(STRICT.values()))
    print(f"\nwinner on VAL: {V['winner']} | S6 trigger (a candidate passes the scorecard on VAL and on CONF-B; read strictly, every item): {'FIRES' if S6 else 'does not fire'}")
    print("candidates that pass the sanity check the standard model fails, at non-inferior AUROC, on VAL and on CONF-B (the brief's wording of S6, reported beside the trigger):", [c for c in CANDS if SANITY[c]] or "none")
    SUMMARY = dict(core_commit=CORE_COMMIT, run=C.CFG.RUN_NAME, smoke=SMOKE_TEST, seeds=SEEDS, addendum_sha256=ADD_SHA, decision_table=TABLE, candidates=CANDS, cards_VAL=V["cards"], cards_CONF_B=CARDS_B,
                   winner_on_VAL=V["winner"], agreement=AGREE, passes_scorecard_strict=STRICT, passes_sanity_at_noninferior_auroc=SANITY, S6_trigger=S6,
                   readouts=dict(VAL=V["readouts"], CONF_B=dict(n=int(len(CONFB["pts"]["a"])), id=CONFB["pts_id"])), evaluation_set=CONFB["tag"], time=time.strftime("%Y-%m-%dT%H:%M:%S"))
    runs.save_json(SUMMARY, "05_summary.json")
    if not SMOKE_TEST:
        runs.save_json(dict(done=True, time=time.strftime("%Y-%m-%dT%H:%M:%S"), core_commit=CORE_COMMIT, addendum_sha256=ADD_SHA, candidates=CANDS), "05_confB_done.json")
        print("CONF-B stage complete; CONF-B is not loaded again. Notebook 04 can now run STAGE = 'EXT' (once).")
    else: print("SMOKE dry run of the CONF-B stage complete (no completion marker written).")

## What to send back

- Stage VAL: the printed output of Cells 2, 4, 6, 7, 8 and 9, or `reports/05_val.json` and `reports/05_methods.json`.
- Stage CONF-B: the printed output of Cells 10 and 11, or `reports/05_summary.json`.

All of it is aggregate. Nothing here was executed before hand-over: expect to fix small things on the first smoke run and send me the traceback.